<a href="https://colab.research.google.com/github/Carinaaa/ML-Learning-Path/blob/intro-LLM/RAG_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import glob
import gradio as gr
from openai import OpenAI
from google.colab import userdata
import requests

In [2]:
api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = api_key # set it as an env var

open_ai = OpenAI()
model = 'gpt-4o-mini'

In [3]:
context = {}
all_employees = ['Alex Chen', 'Alex Harper', 'Alex Thomson', 'Avery Lancaster', 'Emily Carter', 'Emily Tran', 'Jordan Blake', 'Jordan K. Bishop', 'Maxine Thompson', 'Oliver Spencer', 'Samantha Greene', 'Samuel Trenton']
for e in all_employees:
  context[e] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/employees/{e.replace(" ", "%20")}.md').text

employee_context = context.copy()
try:
  os.mkdir('knowedge-base/')
  os.mkdir('knowedge-base/employees')
except FileExistsError:
  print("Dirs already exists.")
for e in all_employees:
  with open(f'knowedge-base/employees/{e}.md', 'w') as f:
    f.write(employee_context[e])

Dirs already exists.


In [4]:
all_products = ['Carllm', 'Homellm', 'Markellm', 'Rellm']
products_context = {}
for p in all_products:
  context[p] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/products/{p}.md').text
  products_context[p] = context[p]

try:
  os.mkdir('knowedge-base/products')
except FileExistsError:
  print("Dirs already exists.")
for p in all_products:
  with open(f'knowedge-base/products/{p}.md', 'w') as f:
    f.write(products_context[p])

Dirs already exists.


In [5]:
company_info = ['about', 'careers', 'overview']
company_context = {}
for e in company_info:
  context[e] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/company/{p}.md').text
  company_context[e] = context[e]
try:
  os.mkdir('knowedge-base/company')
except FileExistsError:
  print("Dirs already exists.")
for e in company_info:
  with open(f'knowedge-base/company/{e}.md', 'w') as f:
    f.write(company_context[e])

Dirs already exists.


In [6]:
contracts_info = ['Contract with Apex Reinsurance for Rellm', 'Contract with Belvedere Insurance for Markellm', 'Contract with BrightWay Solutions for Markellm',
                'Contract with EverGuard Insurance for Rellm', 'Contract with GreenField Holdings for Markellm', 'Contract with GreenValley Insurance for Homellm',
                'Contract with Greenstone Insurance for Homellm','Contract with Pinnacle Insurance Co. for Homellm', 'Contract with Roadway Insurance Inc. for Carllm',
                'Contract with Stellar Insurance Co. for Rellm', 'Contract with TechDrive Insurance for Carllm', 'Contract with Velocity Auto Solutions for Carllm']
contracts_context = {}
for e in contracts_info:
  context[e] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/company/{p.replace(" ", "%20")}.md').text
  contracts_context[e] = context[e]
try:
  os.mkdir('knowedge-base/contracts')
except FileExistsError:
  print("Dirs already exists.")
for e in contracts_info:
  with open(f'knowedge-base/contracts/{e}.md', 'w') as f:
    f.write(contracts_context[e])

Dirs already exists.


In [7]:
context.keys()

dict_keys(['Alex Chen', 'Alex Harper', 'Alex Thomson', 'Avery Lancaster', 'Emily Carter', 'Emily Tran', 'Jordan Blake', 'Jordan K. Bishop', 'Maxine Thompson', 'Oliver Spencer', 'Samantha Greene', 'Samuel Trenton', 'Carllm', 'Homellm', 'Markellm', 'Rellm', 'about', 'careers', 'overview', 'Contract with Apex Reinsurance for Rellm', 'Contract with Belvedere Insurance for Markellm', 'Contract with BrightWay Solutions for Markellm', 'Contract with EverGuard Insurance for Rellm', 'Contract with GreenField Holdings for Markellm', 'Contract with GreenValley Insurance for Homellm', 'Contract with Greenstone Insurance for Homellm', 'Contract with Pinnacle Insurance Co. for Homellm', 'Contract with Roadway Insurance Inc. for Carllm', 'Contract with Stellar Insurance Co. for Rellm', 'Contract with TechDrive Insurance for Carllm', 'Contract with Velocity Auto Solutions for Carllm'])

In [8]:
system_message = "You are an expert in answering accurate questions about Insurellm, the Insurance Tech company. Give brief, accurate answers. If you don't know the answer, say so. Do not make anything up if you haven't been provided with relevant context."

In [9]:
def get_relevant_context(message):
    relevant_context = []
    for context_title, context_details in context.items():
        if context_title.lower() in message.lower():
            relevant_context.append(context_details)
    return relevant_context

In [10]:
def add_context(message):
    relevant_context = get_relevant_context(message)
    if relevant_context:
        message += "\n\nThe following additional context might be relevant in answering this question:\n\n"
        for relevant in relevant_context:
            message += relevant + "\n\n"
    return message

In [11]:
print(add_context("Who is Alex Lancaster?"))

Who is Alex Lancaster?


In [12]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history
    message = add_context(message)
    messages.append({"role": "user", "content": message})

    stream = open_ai.chat.completions.create(model=model, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [13]:
view = gr.ChatInterface(chat, type="messages").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5078c29e8dc9cde7ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
